# Import and configure

In [32]:
import glob
import hashlib
import io
import os
import struct
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from skimage.feature import hog

ROOT = "olympic"
SPLIT_DIR = "train_test_split"
FEATURE_DIR = "features"
K = 10
SIZE = (224, 224)
os.makedirs(FEATURE_DIR, exist_ok=True)

# Build SEQ reader

In [33]:
def read_header(path):
    with open(path, "rb") as f:
        magic, = struct.unpack("<I", f.read(4))
        if magic != 0xFEED:
            raise ValueError(f"not SEQ (magic={magic:#x})")
        f.seek(28)
        version, header_size = struct.unpack("<iI", f.read(8))
        f.seek(548)
        w, h, _, _, _, codec = struct.unpack("<6I", f.read(24))
        n_frames, _, _ = struct.unpack("<3I", f.read(12))
        fps, = struct.unpack("<d", f.read(8))
    return dict(version=version, header_size=header_size, w=w, h=h,
                codec=codec, n_frames=n_frames, fps=fps)


def iter_frames(path, hdr, keep=None):
    ts_bytes = 8 if hdr["version"] >= 5 else 6
    with open(path, "rb") as f:
        f.seek(hdr["header_size"])
        for i in range(hdr["n_frames"]):
            raw = f.read(4)
            if len(raw) < 4:
                raise EOFError(f"file ended before frame {i}")
            size, = struct.unpack("<I", raw)
            if size < 5 or size > 50_000_000:
                raise ValueError(f"cant do frame size {size} at frame {i}")
            jpg = f.read(size - 4)
            if len(jpg) < size - 4:
                raise EOFError(f"truncated JPEG at frame {i}")
            f.read(ts_bytes)
            if keep is None or i in keep:
                yield i, np.array(Image.open(io.BytesIO(jpg)).convert("RGB"))
            else:
                yield i, None


# Construct file list, check for failures, and duplicates

In [34]:
def md5_of(path, chunk=1 << 20):
    m = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            m.update(block)
    return m.hexdigest()


def avg_hash(rgb, n=8):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(g, (n, n), interpolation=cv2.INTER_AREA)
    return "".join("1" if v else "0" for v in (g > g.mean()).flatten())

def decode_check(path, hdr):
    n = hdr["n_frames"]
    sig_idx = [n // 4, n // 2, (3 * n) // 4]
    sig, n_decoded, dims_ok = {}, 0, True
    for i, frame in iter_frames(path, hdr):
        n_decoded += 1
        if frame.shape[:2] != (hdr["h"], hdr["w"]):
            dims_ok = False
        if i in sig_idx:
            sig[i] = avg_hash(frame)
    signature = "".join(sig[i] for i in sig_idx) if all(i in sig for i in sig_idx) else None
    return dict(n_decoded=n_decoded, dims_match=dims_ok, md5=md5_of(path), sig=signature)



def build_index(root):
    rows = []
    for path in sorted(glob.glob(os.path.join(root, "**", "*.seq"), recursive=True)):
        label = os.path.basename(os.path.dirname(path))
        stem = os.path.splitext(os.path.basename(path))[0]
        row = dict(clip_id=f"{label}/{stem}", stem=stem, label=label, path=path,
                   ok=False, error="")
        try:
            hdr = read_header(path)
            row.update(hdr)
            row.update(decode_check(path, hdr))
            if row["n_decoded"] != hdr["n_frames"]:
                row["error"] = "decoded count != header count"
            elif not row["dims_match"]:
                row["error"] = "decoded size != header size"
            elif hdr["n_frames"] < 2:
                row["error"] = "not enough frames for pairing"
            else:
                row["ok"] = True
        except Exception as e:
            row["error"] = repr(e)
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f"no .seq files under {os.path.abspath(root)}; "
                                f"expected {root}")
    for col in ("md5", "sig", "n_decoded", "n_frames", "w", "h", "fps", "codec"):
        if col not in df:
            df[col] = np.nan
    df["dup_exact"] = df.md5.notna() & df.duplicated("md5", keep=False)
    return df


def find_near_duplicates(df, max_bits=12):
    good = df[df.ok & df.sig.notna()].reset_index(drop=True)
    b = np.array([[c == "1" for c in s] for s in good.sig], dtype=np.float32)
    dist = b @ (1 - b).T + (1 - b) @ b.T
    i, j = np.where(np.triu(dist <= max_bits, k=1))
    pairs = pd.DataFrame(dict(
        clip_a=good.clip_id.values[i], clip_b=good.clip_id.values[j],
        bits_differ=dist[i, j].astype(int),
        part_a=good.partition.values[i] if "partition" in good else None,
        part_b=good.partition.values[j] if "partition" in good else None))
    if "partition" in good:
        pairs["crosses_partition"] = pairs.part_a != pairs.part_b
    return pairs


def save_inspection_sheets(df, out_dir="inspection", per_class=3, thumb=(320, 180)):
    os.makedirs(out_dir, exist_ok=True)
    for label, g in df[df.ok].groupby("label"):
        step = max(1, len(g) // per_class)
        tiles = []
        for r in g.iloc[::step].head(per_class).itertuples():
            hdr = read_header(r.path)
            mid = hdr["n_frames"] // 2
            frame = next(f for i, f in iter_frames(r.path, hdr, {mid}) if f is not None)
            tiles.append(cv2.resize(frame, thumb, interpolation=cv2.INTER_AREA))
        sheet = np.hstack(tiles)
        cv2.putText(sheet, label, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        cv2.imwrite(os.path.join(out_dir, f"{label}.png"), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))


# Split dataset via provided splits

In [35]:
def inspect_split(split_dir):
    for p in sorted(glob.glob(os.path.join(split_dir, "**", "*"), recursive=True)):
        if os.path.isfile(p):
            print(p)
            with open(p, errors="replace") as f:
                print("   ", [next(f, "").strip() for _ in range(3)])


def load_split(split_dir):
    mapping = {}
    for p in sorted(glob.glob(os.path.join(split_dir, "**", "*.txt"), recursive=True)):
        name = os.path.basename(p).lower()
        is_train, is_test = "train" in name, "test" in name
        if is_train == is_test:
            print(f"skipping {p}: cannot tell partition from filename")
            continue
        part = "train" if is_train else "test"
        with open(p) as f:
            for line in f:
                tok = line.strip().split()
                if tok:
                    mapping[os.path.splitext(os.path.basename(tok[0]))[0]] = part
    return mapping


def assign_partitions(df, mapping):
    if df.stem.duplicated().any():
        raise ValueError("duplicate clip filenames")
    df["partition"] = df.stem.map(mapping).fillna("unassigned")
    return df

# Motion features

In [36]:
def select_pairs(T, k=K):
    if T < 2:
        return np.array([], dtype=int)
    return np.unique(np.round(np.linspace(0, T - 2, k)).astype(int))

# Resize frames

In [37]:
def resize(rgb):
    return cv2.resize(rgb, SIZE, interpolation=cv2.INTER_AREA)

def to_gray(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def hog_descriptor(gray):
    return hog(gray, orientations=9, pixels_per_cell=(16, 16),
               cells_per_block=(2, 2), block_norm="L2-Hys")

def compute_flow(g0, g1):
    return cv2.calcOpticalFlowFarneback(g0, g1, None, 0.5, 3, 15, 3, 5, 1.2, 0)

def flow_descriptor(flow, grid=(2, 2), bins=8):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    H, W = mag.shape
    out = []
    for gy in range(grid[0]):
        for gx in range(grid[1]):
            ys = slice(gy * H // grid[0], (gy + 1) * H // grid[0])
            xs = slice(gx * W // grid[1], (gx + 1) * W // grid[1])
            m, a = mag[ys, xs], ang[ys, xs]
            hist, _ = np.histogram(a, bins=bins, range=(0, 2 * np.pi), weights=m)
            out.extend(hist / (hist.sum() + 1e-8))
            out.append(m.mean())
    return np.array(out, dtype=np.float32)


def load_resnet(device="cpu"):
    import torch, torchvision
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = torch.nn.Identity()
    return model.eval().to(device)


def resnet_features(model, rgb_frames, device="cpu"):
    import torch
    x = torch.from_numpy(np.stack(rgb_frames)).permute(0, 3, 1, 2).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    with torch.inference_mode():
        return model(((x - mean) / std).to(device)).cpu().numpy().astype(np.float32)

# Features extraction for HOG and ResNet

In [38]:
def process_clip(path, hdr, resnet=None, device="cpu"):
    idx = select_pairs(hdr["n_frames"])
    keep = set(idx.tolist()) | set((idx + 1).tolist())
    got = {i: f for i, f in iter_frames(path, hdr, keep) if f is not None}
    cur = [resize(got[t]) for t in idx]
    nxt = [resize(got[t + 1]) for t in idx]
    g_cur, g_nxt = [to_gray(x) for x in cur], [to_gray(x) for x in nxt]
    out = dict(
        idx=idx,
        hog=np.stack([hog_descriptor(g) for g in g_cur]).astype(np.float32),
        flow=np.stack([flow_descriptor(compute_flow(a, b)) for a, b in zip(g_cur, g_nxt)]))
    if resnet is not None:
        out["resnet"] = resnet_features(resnet, cur, device)
    return out

def feature_path(clip_id):
    return os.path.join(FEATURE_DIR, clip_id.replace("/", "__") + ".npz")

def run_extraction(df, resnet=None, device="cpu", overwrite=False):
    failures = []
    need = {"hog", "flow"} | ({"resnet"} if resnet is not None else set())
    for r in df[df.ok].itertuples():
        p = feature_path(r.clip_id)
        if os.path.exists(p) and not overwrite:
            with np.load(p) as z:
                if need <= set(z.files):
                    continue
        try:
            hdr = read_header(r.path)
            np.savez_compressed(p, **process_clip(r.path, hdr, resnet, device))
        except Exception as e:
            failures.append((r.clip_id, repr(e)))
    return failures


def check_features(df, types=("hog", "flow")):
    dims, problems = {}, []
    for r in df[df.ok].itertuples():
        p = feature_path(r.clip_id)
        if not os.path.exists(p):
            problems.append((r.clip_id, "missing file")); continue
        with np.load(p) as z:
            expected = len(select_pairs(r.n_frames))
            for t in types:
                a = z[t]
                if a.shape[0] != expected:
                    problems.append((r.clip_id, f"{t}: {a.shape[0]} rows, expected {expected}"))
                if not np.isfinite(a).all():
                    problems.append((r.clip_id, f"{t}: NaN/inf"))
                if dims.setdefault(t, a.shape[1]) != a.shape[1]:
                    problems.append((r.clip_id, f"{t}: width {a.shape[1]} != {dims[t]}"))
    counts = df[df.ok].groupby(["label", "partition"]).size().unstack(fill_value=0)
    return problems, counts

# Save features per clip
clip-level pooling (1 vector per clip via 10 per-frame vectors)

In [39]:
def pool(x, mode, segments=3):
    if mode == "mean":
        return x.mean(0)
    if mode == "meanstd":
        return np.concatenate([x.mean(0), x.std(0)])
    if mode == "segments":
        seg = np.linspace(0, segments, len(x), endpoint=False).astype(int)
        return np.concatenate([x[seg == s].mean(0) if (seg == s).any() else x.mean(0)
                               for s in range(segments)])
    raise ValueError(mode)


def build_blocks(df, partition, types, mode):
    blocks, labels = {t: [] for t in types}, []
    for r in df[(df.partition == partition) & df.ok].itertuples():
        with np.load(feature_path(r.clip_id)) as z:
            for t in types:
                blocks[t].append(pool(z[t], mode))
        labels.append(r.label)
    return {t: np.stack(v) for t, v in blocks.items()}, np.array(labels)


def fit_scalers(train_blocks):
    out = {}
    for t, b in train_blocks.items():
        sd = b.std(0)
        out[t] = (b.mean(0), np.where(sd < 1e-6, 1.0, sd))
    return out


def apply_scalers(blocks, scalers):
    parts = []
    for t, b in blocks.items():
        mu, sd = scalers[t]
        parts.append((b - mu) / sd / np.sqrt(b.shape[1]))
    return np.concatenate(parts, axis=1)

# Execute

In [46]:
df = build_index(ROOT)
print(df.groupby("label").ok.agg(["size", "sum"]))
print(df[~df.ok][["clip_id", "error"]])

                  size  sum
label                      
basketball_layup    50    0
                                     clip_id  \
0   basketball_layup/-9t7-hXcl4U_00648_00780   
1   basketball_layup/-9t7-hXcl4U_01168_01301   
2   basketball_layup/17XU-siN32A_00017_00240   
3   basketball_layup/2s5XbNYB4vs_00205_00355   
4   basketball_layup/2s5XbNYB4vs_00408_00622   
5   basketball_layup/2s5XbNYB4vs_00678_00838   
6   basketball_layup/2s5XbNYB4vs_01578_01728   
7   basketball_layup/51H_RVlF2Xk_01482_01608   
8   basketball_layup/51H_RVlF2Xk_01639_01755   
9   basketball_layup/6ciUgtDAe0k_01781_01939   
10  basketball_layup/6qszUMe_7ro_00323_00464   
11  basketball_layup/6qszUMe_7ro_00537_00654   
12  basketball_layup/6qszUMe_7ro_01501_01616   
13  basketball_layup/6qszUMe_7ro_01856_01986   
14  basketball_layup/8eGmno0lZKM_00001_00169   
15  basketball_layup/8eGmno0lZKM_00255_00426   
16  basketball_layup/9jsLPd4_gZ0_00001_00054   
17  basketball_layup/A5oVf8-CoVY_00010_00110   
18  

In [47]:

print(df[["clip_id", "ok", "error"]].head(10))

                                    clip_id     ok  \
0  basketball_layup/-9t7-hXcl4U_00648_00780  False   
1  basketball_layup/-9t7-hXcl4U_01168_01301  False   
2  basketball_layup/17XU-siN32A_00017_00240  False   
3  basketball_layup/2s5XbNYB4vs_00205_00355  False   
4  basketball_layup/2s5XbNYB4vs_00408_00622  False   
5  basketball_layup/2s5XbNYB4vs_00678_00838  False   
6  basketball_layup/2s5XbNYB4vs_01578_01728  False   
7  basketball_layup/51H_RVlF2Xk_01482_01608  False   
8  basketball_layup/51H_RVlF2Xk_01639_01755  False   
9  basketball_layup/6ciUgtDAe0k_01781_01939  False   

                                           error  
0  ValueError('cant do frame size 0 at frame 1')  
1  ValueError('cant do frame size 0 at frame 1')  
2  ValueError('cant do frame size 0 at frame 1')  
3  ValueError('cant do frame size 0 at frame 1')  
4  ValueError('cant do frame size 0 at frame 1')  
5  ValueError('cant do frame size 0 at frame 1')  
6  ValueError('cant do frame size 0 at frame 1')

In [41]:
df = assign_partitions(df, load_split(SPLIT_DIR))
print(df.partition.value_counts())

skipping train_test_split/test/basketball layup.txt: cannot tell partition from filename
skipping train_test_split/test/bowling.txt: cannot tell partition from filename
skipping train_test_split/test/clean and jerk.txt: cannot tell partition from filename
skipping train_test_split/test/discus throw.txt: cannot tell partition from filename
skipping train_test_split/test/hammer throw.txt: cannot tell partition from filename
skipping train_test_split/test/high jump.txt: cannot tell partition from filename
skipping train_test_split/test/javelin throw.txt: cannot tell partition from filename
skipping train_test_split/test/long jump.txt: cannot tell partition from filename
skipping train_test_split/test/platform 10m.txt: cannot tell partition from filename
skipping train_test_split/test/pole vault.txt: cannot tell partition from filename
skipping train_test_split/test/shot put.txt: cannot tell partition from filename
skipping train_test_split/test/snatch.txt: cannot tell partition from filen

In [42]:
pairs = find_near_duplicates(df)
print(pairs[pairs.get("crosses_partition", False)] if len(pairs) else "no duplicates")
print(df[df.dup_exact][["clip_id", "md5"]])

TypeError: tri() missing 1 required positional argument: 'N'

In [43]:
df.drop(columns=["sig"]).to_csv("file_list.csv", index=False)
save_inspection_sheets(df)

In [44]:
print(run_extraction(df))
problems, counts = check_features(df)
print(problems or "all feature files consistent"); print(counts)

types = ["hog", "flow"]
tr, ytr = build_blocks(df, "train", types, "mean")
te, yte = build_blocks(df, "test", types, "mean")
sc = fit_scalers(tr)
Xtr, Xte = apply_scalers(tr, sc), apply_scalers(te, sc)
print(Xtr.shape, Xte.shape)

[]
all feature files consistent
Empty DataFrame
Columns: []
Index: []


ValueError: need at least one array to stack